# What Warsaw diners really think about Polish restaurants beyond the star rating

**Text and Social Media Mining** · University of Warsaw · Winter semester 2026/27 · Mateo Serrato

Heavy or non-repeatable steps (scraping, model training) live in scripts under `src/`; this notebook imports
their functions and saved outputs to present the decisions and results. See the README to reproduce the project.

## 1. Introduction

### Purpose

Star ratings on Google Maps are so compressed (84% of reviews are 5★) that they barely distinguish one restaurant
from another. This project uses text mining on 3,519 Google Maps reviews of 52 Polish restaurants in Warsaw to
identify **what diners actually praise and complain about**, and to examine how this differs by **area, price level
and reviewer language** (Polish vs. English). The goal is to turn unstructured reviews into **specific, validated
insights** a restaurant owner could act on: which aspects drive complaints, and how a restaurant compares with its
competitors.

### Main assumptions

These are my beliefs before looking at the text, written as a diner. Each one will be tested in the analysis.

**A1. Food is the deal-breaker.** You go to a restaurant to eat. If the food is great and the price is fair, slow
service is easy to forgive; if the service is great but the food is bad, people will complain.
*Test:* which aspect dominates the negative sentences in bad reviews, and which of Google's sub-ratings
(food, service, atmosphere) best predicts the overall stars.

**A2. Price is judged as value for money.** A high price is fine if the food and the service are good: you pay for
what you get. Price becomes a complaint when something else goes wrong.
*Test:* price complaints appear together with food or service complaints more often than on their own.

**A3. Stars hide minor complaints.** Most people give 5 stars when nothing went seriously wrong, not when everything
was perfect. A small inconvenience is not worth taking a star away, but it still shows up in the text.
*Test:* share of 4–5★ reviews that contain at least one negative sentence.

**A4. Locals are the more demanding judges of the food.** A tourist has no baseline for Polish food and judges it
against their own expectations; a Pole has eaten it all their life and knows what it should taste like. If a Mexican
tells you a taco place is great, another Mexican will trust it more than someone from Japan would.
*Test:* Polish-language reviews contain more negative food sentences and are more specific (name dishes, compare
with home cooking) than English-language reviews.

**A5. Expensive restaurants get more price complaints,** especially when something else fails, like slow service
or a wrong order.
*Test:* rate of price complaints by price level, and its overlap with service complaints.

**A6. Tourist traps get exposed by locals.** Locals know what a restaurant is worth, so in the city centre Polish
reviews should be more critical than English ones, and more than in other areas.
*Test:* the gap in sentiment between Polish and English reviews is largest in Śródmieście.

**Exploratory question (no prior belief): how do praise and complaints change over time?**

## 2. Data

The dataset is built in four steps: find candidate restaurants on Google Maps, select a sample with fixed rules,
collect their reviews, and clean them. Scraped data is frozen and dated (13 November 2026); everything after it
is deterministic.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import select_restaurants as sel
import clean

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

### 2.1 Source and search

Reviews come from **Google Maps**, the platform with the most reviews for Warsaw restaurants. Candidate restaurants
were found with the open-source scraper [`gosom/google-maps-scraper`](https://github.com/gosom/google-maps-scraper),
using three Polish phrasings (*restauracja polska*, *kuchnia polska*, *pierogarnia*) in each of the five districts.
Milk bars (*bar mleczny*) were deliberately not searched: they are subsidised, counter-service canteens with a
different business model and are not competitors of a Polish restaurant.

In [2]:
queries = (ROOT / "data/raw/gosom_queries.txt").read_text(encoding="utf-8").splitlines()
print(len(queries), "search queries")
print(*queries, sep="\n")

15 search queries
restauracja polska Śródmieście Warszawa
kuchnia polska Śródmieście Warszawa
pierogarnia Śródmieście Warszawa
restauracja polska Mokotów Warszawa
kuchnia polska Mokotów Warszawa
pierogarnia Mokotów Warszawa
restauracja polska Praga-Północ Warszawa
kuchnia polska Praga-Północ Warszawa
pierogarnia Praga-Północ Warszawa
restauracja polska Praga-Południe Warszawa
kuchnia polska Praga-Południe Warszawa
pierogarnia Praga-Południe Warszawa
restauracja polska Wola Warszawa
kuchnia polska Wola Warszawa
pierogarnia Wola Warszawa


### 2.2 Restaurant selection

`src/select_restaurants.py` applies fixed rules, with no hand-picking:

1. In one of the target districts (Google's `borough` field).
2. A Polish restaurant: main category *Kuchnia polska* or *Pierogarnia*, or a generic restaurant explicitly tagged
   *Kuchnia polska* (*Pierogarnia* alone is not enough there: Google also uses it for Asian dumpling places).
3. Not a milk bar (checked in categories **and** name) and not closed.
4. At least 100 Google reviews, so every restaurant has enough text to analyse.
5. Praga-Północ and Praga-Południe are merged into **Praga**: Praga-Północ alone has only 9 eligible restaurants.
6. A **random** sample of 13 per area (seed 42). Random rather than "most popular", so central areas are not
   represented only by famous tourist restaurants.

In [3]:
places = sel.load_places(sel.RAW)
eligible = sel.filter_eligible(places).copy()
eligible["area"] = eligible["district"].map(sel.DISTRICT_TO_AREA)
eligible["area"].value_counts().reindex(sel.AREAS).rename("eligible restaurants")

candidates              494
in target districts     480
Polish restaurant       146
not a milk bar          139
not closed              134
>= 100 reviews          116


area
Śródmieście    56
Mokotów        19
Wola           19
Praga          22
Name: eligible restaurants, dtype: int64

**Reproducibility check:** re-running the selection from the frozen gosom file gives exactly the saved list.

In [4]:
selected = pd.read_csv(sel.OUT)
resampled = sel.sample_per_area(eligible)
assert set(resampled["place_id"]) == set(selected["place_id"])
print(f"OK: {len(selected)} restaurants, identical to data/processed/selected_restaurants.csv")

OK: 52 restaurants, identical to data/processed/selected_restaurants.csv


In [5]:
(selected.groupby("area")
    .agg(restaurants=("name", "size"),
         median_google_reviews=("review_count", "median"),
         mean_rating=("rating", "mean"),
         min_rating=("rating", "min"))
    .reindex(sel.AREAS).round(2))

,restaurants,median_google_reviews,mean_rating,min_rating
area,,,,
Śródmieście,13,1816.0,4.63,4.3
Mokotów,13,579.0,4.62,4.3
Wola,13,410.0,4.62,4.0
Praga,13,461.0,4.60,4.2


Even before reading any text, the ratings are compressed: almost every restaurant sits between 4.3 and 4.9 stars.

### 2.3 Review collection

`src/collect.py` sends the 52 restaurant URLs to the Apify actor *Google Maps Reviews Scraper*:

- the **100 newest** reviews per restaurant: an unbiased, current sample (Google's "most relevant" order is an
  undocumented ranking);
- **no reviewer data** requested (`personalData: False`): names, IDs and photos are never downloaded;
- a hard spending cap. The real run cost $3.11.

In [6]:
items = json.loads(clean.RAW.read_text(encoding="utf-8"))
raw = pd.DataFrame(items)
raw["has_text"] = raw["text"].fillna("").str.strip() != ""
raw["date"] = pd.to_datetime(raw["publishedAtDate"])

print(f"{len(raw)} reviews · {raw['placeId'].nunique()} restaurants · "
      f"{raw.groupby('placeId').size().min()}-{raw.groupby('placeId').size().max()} per restaurant")
print(f"unique review IDs: {raw['reviewId'].nunique()}")
print(f"reviewer names / IDs present: {raw['name'].notna().sum()} / {raw['reviewerId'].notna().sum()}")
print(f"dates: {raw['date'].min():%Y-%m-%d} to {raw['date'].max():%Y-%m-%d}")

5200 reviews · 52 restaurants · 100-100 per restaurant
unique review IDs: 5200
reviewer names / IDs present: 0 / 0
dates: 2017-11-07 to 2026-11-13


**First finding: who writes?** Star distribution of reviews with and without text (%).

In [7]:
t = pd.crosstab(raw["has_text"].map({True: "with text", False: "stars only"}), raw["stars"], normalize="index").mul(100).round(1)
t["reviews"] = raw["has_text"].map({True: "with text", False: "stars only"}).value_counts()
t

stars,1,2,3,4,5,reviews
has_text,,,,,,
stars only,2.7,1.3,2.8,9.9,83.3,1508
with text,5.0,2.3,3.2,5.7,83.9,3692


84% of all reviews are 5★, with or without text, so stars barely separate restaurants. Diners who leave text are
almost twice as likely to give 1★: dissatisfied customers explain themselves. This motivates analysing the text
rather than the stars.

### 2.4 Cleaning

`src/clean.py` produces one row per review:

- keeps only the fields used in the analysis and takes area / district from our own restaurant list;
- removes `<br>` tags and extra whitespace;
- parses the price per person (*"80–100 zł"* → 90; *"Ponad 200 zł"* → 200);
- **drops reviews with fewer than 3 words.** The analysis is about text, and 1–2 word reviews (*"Ok"*, *"Super"*,
  *"Polecam!"*) name no aspect; 155 of those 173 are 5★. Reviews with 3–4 words are kept because they already carry
  aspects (*"Przemiła obsługa, pyszne jedzenie!"*, *"Smaczne, ale coraz drożej."*);
- detects the language with **lingua** and groups it into `pl` / `en` / `other`. Google's own label is kept to cross-check.

In [8]:
n_words = raw["text"].fillna("").map(clean.clean_text).str.split().str.len()
pd.Series({
    "all reviews": len(raw),
    "with text": int((n_words > 0).sum()),
    f">= {clean.MIN_WORDS} words (analysed)": int((n_words >= clean.MIN_WORDS).sum()),
}, name="reviews").to_frame()

,reviews
all reviews,5200
with text,3692
>= 3 words (analysed),3519


In [9]:
reviews = pd.read_csv(clean.OUT, parse_dates=["date"])
print(reviews.shape)
print(f"lingua agrees with Google's language label on {(reviews['lang'] == reviews['lang_google']).mean():.1%} of reviews")
reviews["lang_group"].value_counts().to_frame("reviews")

(3519, 18)
lingua agrees with Google's language label on 99.1% of reviews


,reviews
lang_group,
pl,2654
en,514
other,351


In [10]:
reviews.drop(columns=["text", "place_id", "review_id"]).head()

,restaurant,area,district,date,stars,n_words,lang,lang_group,lang_google,food,service,atmosphere,price_label,price_pln,owner_replied
0,Dom na Chocimskiej,Mokotów,Mokotów,2026-11-12,5,5,pl,pl,pl,5.0,5.0,5.0,NaN,NaN,False
1,Dom na Chocimskiej,Mokotów,Mokotów,2026-11-08,5,15,pl,pl,pl,5.0,5.0,5.0,80–100 zł,90.0,False
2,Dom na Chocimskiej,Mokotów,Mokotów,2026-11-08,5,10,pl,pl,pl,5.0,5.0,5.0,140–160 zł,150.0,False
3,Dom na Chocimskiej,Mokotów,Mokotów,2026-11-06,5,202,pl,pl,pl,5.0,5.0,5.0,100–120 zł,110.0,True
4,Dom na Chocimskiej,Mokotów,Mokotów,2026-11-06,5,32,en,en,en,5.0,5.0,5.0,Ponad 200 zł,200.0,False


### 2.5 Exploratory data analysis

*Next step.*

## 3. Modelling

*To do: split reviews into sentences, discover aspects (BERTopic on multilingual embeddings, LDA baseline),
sentence-level sentiment (XLM-RoBERTa), aspect × sentiment table, comparisons by area / price / language.*

## 4. Evaluation

*To do: agreement with Google's food / service / atmosphere sub-ratings, hand-labelled sample of sentences
(target ≥80% agreement on aspect + sentiment), topic coherence.*

## 5. Summary

*To do: was the purpose achieved, were the assumptions examined, conclusions.*